# Testing Scraper on GDELT Data

### Imports and Notebook Settings


In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
import os 
import re
from pathlib import Path

In [58]:
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
# import seaborn as sns 
import plotly.express as px
import pprint 

In [59]:
import requests
import trafilatura
from bs4 import BeautifulSoup
from datetime import datetime, timezone

In [60]:
ROOT = Path.cwd().parent

In [61]:
style_path = ( ROOT 
              / 'notebooks' 
              / 'styler.mplstyle'
              )
plt.style.use(style_path)

In [62]:
data_path = (ROOT 
             / 'data'
             / 'silver'
             / 'source=gdelt'
             /  'records.parquet'
             )

In [63]:
df = pd.read_parquet(data_path)

In [64]:
df.drop(columns=['raw'])

,record_id,source,source_type,title,text,published_at,retrieved_at,url,region,categories,metadata
0,20260420130000-579,gdelt,api,'The absolute edge of precedent': FERC prepares to take on data centers,,2026-04-20T13:00:00+00:00,2026-06-18T14:49:51.869112+00:00,https://www.eenews.net/articles/the-absolute-edge-of-precedent-ferc-prepares-to-take-on-data-centers/,"2#NEW JERSEY, UNITED STATES#US#USNJ#40.314#-74.5089#NJ","[EPU_POLICY, EPU_POLICY_WHITE_HOUSE, TAX_FNCACT, TAX_FNCACT_REGULATORS, EPU_POLICY_REGULATORY, PROTEST, STRIKE, ECON_ELECTRICALDEMAND, TAX_FNCACT_SECRETARY, ECON_ELECTRICALGRID, TECH_SUPERCOMPUTING, USPEC_POLICY1, EPU_ECONOMY, EPU_ECONOMY_HISTORIC, LEADER, TAX_FNCACT_PRESIDENT, USPEC_POLITICS_GENERAL1, TRIAL, TAX_FNCACT_ATTORNEY, LEGISLATION, EPU_POLICY_LAW, EPU_CATS_MIGRATION_FEAR_FEAR, WB_696_PUBLIC_SECTOR_MANAGEMENT, WB_831_GOVERNANCE, WB_1921_PRIVATE_SECTOR_DEVELOPMENT, WB_346_COMPETITIVE_INDUSTRIES, WB_818_INDUSTRY_POLICY_AND_REAL_SECTORS, WB_1281_MANUFACTURING, TAX_ETHNICITY, TAX_ETHNICITY_AMERICAN, SCIENCE, SOC_INNOVATION, TAX_FNCACT_REGULATOR, TAX_FNCACT_COMMISSIONERS, TAX_FNCACT_SPOKESPERSON, EPU_POLICY_POLICY, ELECTION, WB_137_WATER, TAX_ECON_PRICE, ECON_COST_OF_LIVING, ECON_ELECTRICALPRICE, DEMOCRACY, TAX_POLITICAL_PARTY, TAX_POLITICAL_PARTY_REPUBLICAN, EPU_POLICY_SPENDING, WB_678_DIGITAL_GOVERNMENT, WB_2944_SERVERS, WB_671_STORAGE_MANAGEMENT, WB_667_ICT_INFRASTRUCTURE, WB_672_NETWORK_MANAGEMENT, WB_133_INFORMATION_AND_COMMUNICATION_TECHNOLOGIES, WB_675_WEB_FARM, WB_674_SHARED_INFRASTRUCTURE, TAX_FNCACT_DIRECTOR, EDUCATION, SOC_POINTSOFINTEREST, SOC_POINTSOFINTEREST_SCHOOL, TAX_FNCACT_POLITICIANS, BAN, TAX_FNCACT_REPRESENTATIVES, TAX_FNCACT_POLITICO, UNGP_POLITICAL_FREEDOMS, ECON_ELECTRICALGENERATION, WB_508_POWER_SYSTEMS, WB_507_ENERGY_AND_EXTRACTIVES, ENV_COAL, UNGP_FORESTS_RIVERS_OCEANS, WB_509_NUCLEAR_ENERGY, ENV_SOLAR, WB_525_RENEWABLE_ENERGY, WB_528_SOLAR_ENERGY, TAX_FNCACT_GOVERNORS, WB_2048_COMPENSATION_CAREERS_AND_INCENTIVES, WB_723_PUBLIC_ADMINISTRATION, WB_724_HUMAN_RESOURCES_FOR_PUBLIC_SECTOR, WB_698_TRADE, WB_840_JUSTICE, TAX_FNCACT_PROFESSOR, EPU_POLICY_REGULATION, EPU_CATS_REGULATION, TAX_FNCACT_CEO, CRISISLEX_T11_UPDATESSYMPATHY, TAX_FNCACT_EXECUTIVE, TAX_FNCACT_OPERATOR, AFFECT, TAX_FNCACT_EXECUTIVE_DIRECTOR, ALLIANCE, WB_694_BROADCAST_AND_MEDIA, MEDIA_SOCIAL, WB_652_ICT_APPLICATIONS, WB_662_SOCIAL_MEDIA, WB_658_ENTERPRISE_APPLICATIONS]","{'source_common_name': 'eenews.net', 'gdelt_timestamp': '20260420130000', 'organizations': ['american electric power', 'edison electric institute', 'yale law school', 'dominion energy', 'energy law', 'electricity consumers resource council', 'william mary law school', 'energy department', 'energy regulatory commission', 'microsoft', 'oracle', 'national association of regulatory utility commissioners', 'white house', 'google', 'christie', 'electricity customer alliance', 'entergy', 'energy bar association'], 'persons': ['chris wright', 'vinson elkins', 'mark christie', 'laura swett', 'glenn youngkin', 'donald trump', 'karen onaran', 'justice brandeis', 'josh shapiro', 'vince duane', 'jeff dennis', 'joshua macey'], 'locations': ['2#NEW JERSEY, UNITED STATES#US#USNJ#40.314#-74.5089#NJ', 'GEORGIA, UNITED STATES', '2#TEXAS, UNITED STATES#US#USTX#31.106#-97.6475#TX', 'TEXAS, UNITED STATES', '2#VIRGINIA, UNITED STATES#US#USVA#37.768#-78.2057#VA', 'US', '2#GEORGIA, UNITED STATES#US#USGA#32.9866#-83.6487#GA', '2#OHIO, UNITED STATES#US#USOH#40.3736#-82.7755#OH', '3#WHITE HOUSE, DISTRICT OF COLUMBIA, UNITED STATES#US#USDC#38.8951#-77.0364#531871', 'USNJ', 'USPA', 'USGA', '1#UNITED STATES#US#US#39.828175#-98.5795#US', 'OHIO, UNITED STATES', 'USDC', 'USOH', 'NEW JERSEY, UNITED STATES', '2#PENNSYLVANIA, UNITED STATES#US#USPA#40.5773#-77.264#PA', 'PENNSYLVANIA, UNITED STATES', 'USIN', 'USTX', 'UNITED STATES', 'VIRGINIA, UNITED STATES', 'WHITE HOUSE, DISTRICT OF COLUMBIA, UNITED STATES', '2#INDIANA, UNITED STATES#US#USIN#39.8647#-86.2604#IN', 'INDIANA, UNITED STATES',

In [65]:
for i in df['title'].head(30):
    print(i)

'The absolute edge of precedent': FERC prepares to take on data centers
American Electric Power (NASDAQ:AEP) Price Target Raised to $144.00 at Wells Fargo & Company
Wreck knocks out power to hundreds in Orange overnight
Michigan Feared Cheboygan Dam Danger Before Rains Pushed it to Brink
Duke Energy: 100-Year Dividend Track Record — But $103B Capex Is Compressing The Credit Cushion - Duke En
Meta's $27 Billion Data Center Places the Spotlight on Natural Gas Stocks
Xcel: power shutoff 'possible' Wednesday amid wildfire risk | Western Colorado
Analysts' Recent Ratings Updates for American Electric Power (AEP)
Truist Financial Begins Coverage on American Electric Power (NASDAQ:AEP)
Powering Reliability: Duke Energy lineworkers sharpen skills at Carolinas Lineman's regional rodeo
Consumers Energy installs new weather stations to improve storm response
Consumers Energy launches weather station network in five Michigan counties
Alliant Energy: New wind project moves forward in Minnesota
Cons

In [83]:
import nest_asyncio
nest_asyncio.apply()


In [90]:
from datetime import datetime, timezone
import json

import aiohttp
import trafilatura
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


def categorize_status(status: int | None) -> str:
    if status == 403:
        return "forbidden"
    if status == 404:
        return "not_found"
    if status == 429:
        return "rate_limited"
    if status is not None and 500 <= status < 600:
        return "server_error"
    if status is not None:
        return "http_error"
    return "unknown"


async def fetch_html(
    session: aiohttp.ClientSession,
    url: str,
    timeout: int = 15,
) -> dict:
    try:
        async with session.get(url, timeout=timeout) as response:
            text = await response.text(errors="ignore")

            if response.status >= 400:
                return {
                    "success": False,
                    "html": None,
                    "status_code": response.status,
                    "error_type": categorize_status(response.status),
                    "error_message": f"HTTP {response.status}",
                    "method": "aiohttp",
                }

            return {
                "success": True,
                "html": text,
                "status_code": response.status,
                "error_type": None,
                "error_message": None,
                "method": "aiohttp",
            }

    except TimeoutError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "timeout",
            "error_message": str(exc),
            "method": "aiohttp",
        }

    except aiohttp.ClientConnectionError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "connection_error",
            "error_message": str(exc),
            "method": "aiohttp",
        }

    except Exception as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "unknown",
            "error_message": str(exc),
            "method": "aiohttp",
        }


async def fetch_html_with_playwright(
    url: str,
    timeout: int = 30000,
) -> dict:
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)

            page = await browser.new_page(
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/137.0.0.0 Safari/537.36"
                )
            )

            response = await page.goto(
                url,
                wait_until="networkidle",
                timeout=timeout,
            )

            html = await page.content()
            status_code = response.status if response else None

            await browser.close()

        return {
            "success": True,
            "html": html,
            "status_code": status_code,
            "error_type": None,
            "error_message": None,
            "method": "playwright",
        }

    except PlaywrightTimeoutError as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "playwright_timeout",
            "error_message": str(exc),
            "method": "playwright",
        }

    except Exception as exc:
        return {
            "success": False,
            "html": None,
            "status_code": None,
            "error_type": "playwright_error",
            "error_message": str(exc),
            "method": "playwright",
        }


def extract_with_trafilatura(
    html: str,
    url: str,
    extractor: str,
) -> dict:
    result = trafilatura.extract(
        html,
        url=url,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_tables=False,
    )

    if result is None:
        return {
            "success": False,
            "extractor": extractor,
            "title": None,
            "author": None,
            "date": None,
            "text": "",
            "error_type": "parse_failed",
            "error_message": "trafilatura returned None",
        }

    parsed = json.loads(result)
    text = parsed.get("text", "") or ""

    return {
        "success": True,
        "extractor": extractor,
        "title": parsed.get("title"),
        "author": parsed.get("author"),
        "date": parsed.get("date"),
        "text": text,
        "error_type": None,
        "error_message": None,
    }


async def scrape_article(
    session: aiohttp.ClientSession,
    url: str,
) -> dict:
    retrieved_at = datetime.now(timezone.utc).isoformat()

    fetch_result = await fetch_html(session, url)

    if (
        not fetch_result["success"]
        and fetch_result["error_type"] == "forbidden"
    ):
        fetch_result = await fetch_html_with_playwright(url)

    if not fetch_result["success"]:
        return {
            "success": False,
            "url": url,
            "status_code": fetch_result["status_code"],
            "error_type": fetch_result["error_type"],
            "error_message": fetch_result["error_message"],
            "fetch_method": fetch_result["method"],
            "extractor": None,
            "title": None,
            "author": None,
            "date": None,
            "text": "",
            "text_length": 0,
            "retrieved_at": retrieved_at,
        }

    extracted = extract_with_trafilatura(
        html=fetch_result["html"],
        url=url,
        extractor=f"trafilatura_after_{fetch_result['method']}",
    )

    return {
        "success": extracted["success"],
        "url": url,
        "status_code": fetch_result["status_code"],
        "error_type": extracted["error_type"],
        "error_message": extracted["error_message"],
        "fetch_method": fetch_result["method"],
        "extractor": extracted["extractor"],
        "title": extracted["title"],
        "author": extracted["author"],
        "date": extracted["date"],
        "text": extracted["text"],
        "text_length": len(extracted["text"]),
        "retrieved_at": retrieved_at,
    }

In [92]:
import pandas as pd
import aiohttp
import asyncio


async def scrape_urls(urls: list[str]) -> pd.DataFrame:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
    }

    async with aiohttp.ClientSession(headers=headers) as session:
        results = [
            await scrape_article(session, url)
            for url in urls
        ]

    return pd.DataFrame(results)



In [101]:
urls = df["url"].dropna().head(100).tolist()

results_df = await scrape_urls(urls)

In [102]:
results_df[
    [
        "success",
        "url",
        "title",
        "date",
        "text_length",
        "extractor",
    ]
]

,success,url,title,date,text_length,extractor
0,True,https://www.eenews.net/articles/the-absolute-edge-of-precedent-ferc-prepares-to-take-on-data-centers/,‘The absolute edge of precedent’: FERC prepares to take on data centers,2026-04-20,13020,trafilatura_after_playwright
1,True,https://www.themarketsdaily.com/2026/04/20/american-electric-power-nasdaqaep-price-target-raised-to-144-00-at-wells-fargo-company.html,American Electric Power (NASDAQ:AEP) Price Target Raised to $144.00 at Wells Fargo & Company,2026-04-20,5933,trafilatura_after_aiohttp
2,True,https://www.12newsnow.com/article/traffic/wreck-knocks-out-power-to-hundreds-in-orange/502-24e43403-5c49-4c2a-948b-bfbd4ca8cbdf,Access Denied,None,300,trafilatura_after_playwright
3,True,https://www.insurancejournal.com/news/midwest/2026/04/20/866603.htm,Michigan Feared Cheboygan Dam Danger Before Rains Pushed it to Brink,2026-04-20,9263,trafilatura_after_aiohttp
4,True,https://www.benzinga.com/Opinion/26/04/51923036/duke-energy-100-year-dividend-track-record-but-103b-capex-is-compressing-the-credit-cushion,Duke Energy: 100-Year Dividend Track Record — But $103B Capex Is Compressing The Credit Cushion - Duke En,2026-04-20,4349,trafilatura_after_aiohttp
...,...,...,...,...,...,...
95,True,https://wbt.com/1644984/appeals-court-wrestles-with-nc-permit-for-mvp-pipeline/,Appeals Court wrestles with NC permit for MVP pipeline,2026-04-28,10223,trafilatura_after_aiohttp
96,False,https://wwmt.com/news/local/consumers-energy-deploys-500-crews-as-outages-peak-at-86000-most-back-by-tuesday-night-greg-salisbury-emmett-urbandale-pennfield-township-power-grid,None,None,0,None
97,True,https://wwmt.com/news/local/consumers-energy-deploys-500-crews-outages-peak-86000-restore-by-tuesday-night-greg-salisbury-emmett-urbandale-pennfield-township-power-grid,"Consumers Energy deploys 500 crews as outages peak at 86K, most restored by Tuesday night",2026-04-28,4071,trafilatura_after_aiohttp
98,True,https://www.fool.com/investing/2026/04/29/energy-stock-showdown-nextera-or-duke-energy-which/?source=iedfolrf0000001,Energy Stock Showdown: NextEra or Duke Energy -- Which One Wins Right Now? | The Motley Fool,2026-04-29,3437,trafilatura_after_aiohttp


In [103]:
success_rate = results_df['success'].sum() / len(results_df)

print(f"{success_rate * 100}% of articles could be scraped")

90.0% of articles could be scraped


In [104]:
results_df["error_type"].value_counts(dropna=False)

error_type
None                  90
not_found              5
playwright_timeout     2
unknown                2
rate_limited           1
Name: count, dtype: int64

In [106]:
pd.set_option("display.max_colwidth", None)

results_df[
    ~results_df["success"]
][
    [
        "url",
        "status_code",
        "error_type",
        "error_message"
    ]
]

,url,status_code,error_type,error_message
7,https://www.dailypolitical.com/2026/04/21/analysts-recent-ratings-updates-for-american-electric-power-aep.html,404.0,not_found,HTTP 404
8,https://www.dailypolitical.com/2026/04/21/truist-financial-begins-coverage-on-american-electric-power-nasdaqaep.html,404.0,not_found,HTTP 404
30,https://www.erienewsnow.com/news/state/ohio-supreme-court-guts-submetering-business-said-to-drive-up-renters-electric-bills/article_7e99e81f-249c-5e71-80e5-5ba1c2aeb0b0.html,404.0,not_found,HTTP 404
44,https://www.powermag.com/duke-energys-robinson-nuclear-plant-gets-nrc-approval-to-operate-until-2050/,NaN,playwright_timeout,"Page.goto: Timeout 30000ms exceeded.\nCall log:\n - navigating to ""https://www.powermag.com/duke-energys-robinson-nuclear-plant-gets-nrc-approval-to-operate-until-2050/"", waiting until ""networkidle""\n"
56,https://kfiz.com/state-regulators-change-we-energies-data-center-rate-proposal-to-protect-customers/,NaN,playwright_timeout,"Page.goto: Timeout 30000ms exceeded.\nCall log:\n - navigating to ""https://kfiz.com/state-regulators-change-we-energies-data-center-rate-proposal-to-protect-customers/"", waiting until ""networkidle""\n"
59,https://finance.yahoo.com/markets/stocks/articles/southern-company-raises-dividend-2-141743349.html,NaN,unknown,"400, message='Got more than 8190 bytes when reading: b""connect-src \'self\' wss://*.finance.yahoo.com/ https://*.cdn.yimg.com https://*.oath.com https://*.ya..."".', url='https://finance.yahoo.com/markets/stocks/articles/southern-company-raises-dividend-2-141743349.html'"
60,https://finance.yahoo.com/sectors/energy/articles/nrc-renews-license-duke-energy-141832333.html,NaN,unknown,"400, message='Got more than 8190 bytes when reading: b""connect-src \'self\' wss://*.finance.yahoo.com/ https://*.cdn.yimg.com https://*.oath.com https://*.ya..."".', url='https://finance.yahoo.com/sectors/energy/articles/nrc-renews-license-duke-energy-141832333.html'"
89,https://www.yahoo.com/news/articles/stein-power-demand-rises-north-211542291.html,429.0,rate_limited,HTTP 429
94,https://www.wlfi.com/news/lafayette-cleanup-underway-after-storms-leave-hundreds-without-power/article_4efb86af-befd-4084-988b-cbaa6b004657.html,404.0,not_found,HTTP 404
96,https://wwmt.com/news/local/consumers-energy-deploys-500-crews-as-outages-peak-at-86000-most-back-by-tuesday-night-greg-salisbury-emmett-urbandale-pennfield-township-power-grid,404.0,not_found,HTTP 404


In [107]:
results_df.columns

Index(['success', 'url', 'status_code', 'error_type', 'error_message',
       'fetch_method', 'extractor', 'title', 'author', 'date', 'text',
       'text_length', 'retrieved_at'],
      dtype='object')